<a href="https://colab.research.google.com/github/prbchinmayi/Deforestration_detection_ResNet50/blob/main/Deforestration_detection_resnet50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#check 1
import torch
print(torch.cuda.is_available())


#check 2
print("python version:",torch.__version__, torch.device("cuda" if torch.cuda.is_available() else "cpu"))
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

True
python version: 2.11.0+cu128 cuda
Tesla T4


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

In [ ]:
from torchvision.datasets import EuroSAT

transform= transforms.Compose([transforms.Resize((224,224)), # ResNet-50 expects 224x224 px
                               transforms.ToTensor(),
                               transforms.Normalize(
                                   mean=[0.485, 0.456, 0.406],
                                   std=[0.229, 0.224, 0.225]
                               )
                             ])

dataset=EuroSAT(root="./data", transform=transform,download=True)
classes=dataset.classes
size=len(dataset)

#dataset info
print(f"no of classes:{len(classes)}")
print(f"Classes:")
for cl in (classes):
  print(" ", cl)
print(f"no of images:{size}")


100%|██████████| 94.3M/94.3M [00:00<00:00, 110MB/s]


no of classes:10
Classes:
  AnnualCrop
  Forest
  HerbaceousVegetation
  Highway
  Industrial
  Pasture
  PermanentCrop
  Residential
  River
  SeaLake
no of images:27000


In [ ]:
#splitting dataset
train_size=int(0.70*size)
val_size=int(0.15*size)
test_size=int(0.15*size)

train_set, val_set, test_set= random_split(dataset,[train_size, val_size, test_size], generator= torch.Generator().manual_seed(42)) #same split each time
print(f"training: {len(train_set)}")
print(f"validation: {len(val_set)}")
print(f"testing: {len(test_set)}")

#dataloaders to divide images in batches for model to process
batch_size=32

train_loader= DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader= DataLoader(val_set, batch_size=batch_size, shuffle=True)
test_loader= DataLoader(test_set, batch_size=batch_size, shuffle=True)

training: 18900
validation: 4050
testing: 4050


In [ ]:
import copy
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model factory based on gene values
def create_candidate_model(unfreeze_depth, dropout_rate, num_classes=10):
    weights = models.ResNet50_Weights.DEFAULT
    model = models.resnet50(weights=weights)

    # Freeze entire backbone
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze layers according to gene
    if unfreeze_depth >= 1:
        for param in model.layer4.parameters():
            param.requires_grad = True
    if unfreeze_depth >= 2:
        for param in model.layer3.parameters():
            param.requires_grad = True
    if unfreeze_depth >= 3:
        for param in model.parameters():
            param.requires_grad = True

    # Build head with evolved dropout
    in_features = model.fc.in_features
    if dropout_rate > 0.05:
        model.fc = nn.Sequential(
            nn.Dropout(p=dropout_rate), nn.Linear(in_features, num_classes)
        )
    else:
        model.fc = nn.Linear(in_features, num_classes)

    return model.to(device)


# Fitness function using a fast proxy budget (2 epochs on 20% of batches)
def compute_fitness(chromosome, train_loader, val_loader, eval_epochs=2):
    unfreeze_depth, lr_head, lr_backbone, dropout, weight_decay = chromosome

    model = create_candidate_model(unfreeze_depth, dropout)
    criterion = nn.CrossEntropyLoss()

    backbone_params = [
        p
        for n, p in model.named_parameters()
        if "fc" not in n and p.requires_grad
    ]
    param_groups = [{"params": model.fc.parameters(), "lr": lr_head}]
    if len(backbone_params) > 0:
        param_groups.append({"params": backbone_params, "lr": lr_backbone})

    optimizer = optim.AdamW(param_groups, weight_decay=weight_decay)

    # Proxy training
    model.train()
    batch_limit = len(train_loader) // 5
    for epoch in range(eval_epochs):
        for i, (imgs, labels) in enumerate(train_loader):
            if i >= batch_limit:
                break
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()

    # Proxy validation evaluation
    model.eval()
    correct, total = 0, 0
    val_limit = len(val_loader) // 5
    with torch.no_grad():
        for i, (imgs, labels) in enumerate(val_loader):
            if i >= val_limit:
                break
            imgs, labels = imgs.to(device), labels.to(device)
            preds = torch.argmax(model(imgs), dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

    return (correct / total) * 100


# GA Operators
def generate_individual():
    return [
        random.randint(0, 3),  # unfreeze_depth (0=fc, 1=+layer4, 2=+layer3, 3=all)
        10 ** random.uniform(-4, -2),  # lr_head (1e-4 to 1e-2)
        10 ** random.uniform(-6, -4),  # lr_backbone (1e-6 to 1e-4)
        random.uniform(0.0, 0.5),  # dropout rate
        10 ** random.uniform(-5, -2),  # weight decay (1e-5 to 1e-2)
    ]


def uniform_crossover(parent_1, parent_2):
    return [
        p1 if random.random() < 0.5 else p2
        for p1, p2 in zip(parent_1, parent_2)
    ]


def mutate_individual(chromosome, rate=0.3):
    c = copy.deepcopy(chromosome)
    if random.random() < rate:
        c[0] = random.randint(0, 3)
    if random.random() < rate:
        c[1] = 10 ** random.uniform(-4, -2)
    if random.random() < rate:
        c[2] = 10 ** random.uniform(-6, -4)
    if random.random() < rate:
        c[3] = random.uniform(0.0, 0.5)
    if random.random() < rate:
        c[4] = 10 ** random.uniform(-5, -2)
    return c

In [ ]:
POPULATION_SIZE = 6
GENERATIONS = 3

population = [generate_individual() for _ in range(POPULATION_SIZE)]
best_overall_score = -1.0
best_overall_config = None

for gen in range(GENERATIONS):
    print(f"\n================ Running Generation {gen + 1}/{GENERATIONS} ================")
    scores = []

    for idx, ind in enumerate(population):
        score = compute_fitness(ind, train_loader, val_loader)
        scores.append(score)
        print(
            f"Candidate {idx + 1} | Unfreeze: {ind[0]}, Head LR: {ind[1]:.5f}, "
            f"Backbone LR: {ind[2]:.6f}, Dropout: {ind[3]:.2f}, WD: {ind[4]:.5f} "
            f"-> Proxy Acc: {score:.2f}%"
        )

        if score > best_overall_score:
            best_overall_score = score
            best_overall_config = ind

    # Elitism: retain the best 2 candidates
    ranked = np.argsort(scores)[::-1]
    new_population = [population[ranked[0]], population[ranked[1]]]

    # Breed remaining slots
    while len(new_population) < POPULATION_SIZE:
        t1, t2 = random.sample(population, 2), random.sample(population, 2)
        p1 = max(t1, key=lambda x: scores[population.index(x)])
        p2 = max(t2, key=lambda x: scores[population.index(x)])
        child = mutate_individual(uniform_crossover(p1, p2))
        new_population.append(child)

    population = new_population

print(f"\nEvolution Finished.")
print(f"Top Validation Accuracy Found: {best_overall_score:.2f}%")
print("Selected Best Configuration:", best_overall_config)


================ Running Generation 1/3 ================
Candidate 1 | Unfreeze: 2, Head LR: 0.00686, Backbone LR: 0.000009, Dropout: 0.40, WD: 0.00045 -> Proxy Acc: 91.75%
Candidate 2 | Unfreeze: 3, Head LR: 0.00010, Backbone LR: 0.000001, Dropout: 0.27, WD: 0.00246 -> Proxy Acc: 76.25%
Candidate 3 | Unfreeze: 2, Head LR: 0.00381, Backbone LR: 0.000026, Dropout: 0.10, WD: 0.00003 -> Proxy Acc: 94.62%
Candidate 4 | Unfreeze: 2, Head LR: 0.00114, Backbone LR: 0.000004, Dropout: 0.31, WD: 0.00149 -> Proxy Acc: 90.75%
Candidate 5 | Unfreeze: 1, Head LR: 0.00086, Backbone LR: 0.000019, Dropout: 0.18, WD: 0.00018 -> Proxy Acc: 93.00%
Candidate 6 | Unfreeze: 3, Head LR: 0.00014, Backbone LR: 0.000002, Dropout: 0.39, WD: 0.00255 -> Proxy Acc: 75.00%

================ Running Generation 2/3 ================
Candidate 1 | Unfreeze: 2, Head LR: 0.00381, Backbone LR: 0.000026, Dropout: 0.10, WD: 0.00003 -> Proxy Acc: 94.62%
Candidate 2 | Unfreeze: 1, Head LR: 0.00086, Backbone LR: 0.000019, Drop

In [ ]:
# Unpack the winning individual
unfreeze_depth, best_head_lr, best_bb_lr, best_dropout, best_wd = (
    best_overall_config
)

# Instantiate the optimal model
model = create_candidate_model(
    unfreeze_depth, best_dropout, num_classes=len(classes)
)

backbone_params = [
    p for n, p in model.named_parameters() if "fc" not in n and p.requires_grad
]
param_groups = [{"params": model.fc.parameters(), "lr": best_head_lr}]
if len(backbone_params) > 0:
    param_groups.append({"params": backbone_params, "lr": best_bb_lr})

criterion = nn.CrossEntropyLoss()
optimiser = optim.AdamW(param_groups, weight_decay=best_wd)

# Full training run
no_epochs = 6
for epoch in range(no_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimiser.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimiser.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = (correct / total) * 100

    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = (val_correct / val_total) * 100
    print(
        f"Epoch [{epoch + 1}/{no_epochs}] | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%"
    )

Epoch [1/6] | Train Acc: 92.33% | Val Acc: 96.74%
Epoch [2/6] | Train Acc: 97.75% | Val Acc: 97.33%
Epoch [3/6] | Train Acc: 98.68% | Val Acc: 97.56%
Epoch [4/6] | Train Acc: 98.98% | Val Acc: 97.53%
Epoch [5/6] | Train Acc: 99.19% | Val Acc: 97.60%
Epoch [6/6] | Train Acc: 99.33% | Val Acc: 97.90%


In [ ]:
model.eval()
test_correct=0
test_total=0

with torch.no_grad():
  for imgs, labels in test_loader:
    imgs=imgs.to(device)
    labels=labels.to(device)

    outputs=model(imgs)

    _, predicted= torch.max(outputs,1)
    test_total= test_total+labels.size(0)
    test_correct= test_correct+(predicted==labels).sum().item()
test_acc=(test_correct/test_total)*100
print(f"testing accuracy: {test_acc:.2f}")

testing accuracy: 98.10


In [ ]:
# user innput testing
from google.colab import files
from PIL import Image
predicted_class=[]
for i in range(2):
  uploaded=files.upload()
  name=list(uploaded.keys())[0]
  img= Image.open(name).convert("RGB")

  tensor=transform(img)
  tensor=tensor.unsqueeze(0)

  tensor=tensor.to(device)

  model.eval()

  with torch.no_grad():
      output = model(tensor)

      probabilities = torch.softmax(output, dim=1)

      confidence, predicted = torch.max(probabilities, 1)

  # Get class name
  predicted_class.append(dataset.classes[predicted.item()])

  # Display result
  print(f"Predicted class for image {i+1}:", predicted_class[i])
  print("Confidence:", f"{confidence.item() * 100:.2f}%\n")

if(predicted_class[0]=="Forest" and predicted_class[1]!="Forest"):
  print("deforestration detected.")
  print(f"change from {predicted_class[0]}->{predicted_class[1]}")
else:
  print("no deforestration detected.")

Saving image1.jpg to image1 (3).jpg
Predicted class for image 1: Forest
Confidence: 100.00%



Saving 2011.png to 2011 (1).png
Predicted class for image 2: AnnualCrop
Confidence: 52.05%

deforestration detected.
change from Forest->AnnualCrop
